# Pilar I — Invariantes físicas y clustering interpretable

**Objetivo.** Construir invariantes ciberfísicas (descartan el nivel absoluto y dejan la física),
clusterizarlas con el mejor pipeline y **perfilar** cada cluster con z-score para interpretar qué agrupa.

Complementa `NOSUPERVISADO.ipynb` **sin modificarlo** y reutiliza los helpers de `src2`.

## 1. Entorno y carga

Se reutilizan `eliminar_columnas`, `imputar_media` y `separar_variables` de `src2`,
con los mismos ocho escenarios del trabajo.

In [ ]:
# --- Entorno y carga: se reutilizan los helpers de src2 ---
import sys
from pathlib import Path

# Localizar la raíz del repo y poner Colab/ en el path para importar src2.
RUTA = Path.cwd()
if not (RUTA / "datasets").exists():
    RUTA = RUTA.parent
for ruta in (RUTA, RUTA / "Colab"):
    if str(ruta) not in sys.path:
        sys.path.insert(0, str(ruta))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import (adjusted_rand_score, davies_bouldin_score,
                             silhouette_score)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import RobustScaler

from src2.data import eliminar_columnas, imputar_media
from src2.split import separar_variables

SEED = 42
BASE = RUTA / "datasets"

# Un archivo por escenario (S1-alt usa el nombre sin guion).
ESCENARIOS = {
    "S0": "datasetS0.csv", "S1": "datasetS1.csv", "S1-alt": "datasetS1alt.csv",
    "S2": "datasetS2.csv", "S3": "datasetS3.csv", "S4": "datasetS4.csv",
    "S5": "datasetS5.csv", "S6": "datasetS6.csv",
}

# Carga, limpieza e imputación con los mismos helpers que NOSUPERVISADO.
brutos = {k: pd.read_csv(BASE / archivo) for k, archivo in ESCENARIOS.items()}
X, Y = {}, {}
for nombre, bloque in brutos.items():
    limpio = imputar_media(eliminar_columnas(bloque))
    X[nombre], Y[nombre] = separar_variables(limpio)

print("escenarios:", list(X))
print("features por escenario:", {k: v.shape[1] for k, v in X.items()})

## 2. Invariantes ciberfísicas

Tres transformaciones que eliminan el nivel absoluto (lo que arrastra el tiempo) y conservan la física:

- **ΔSOC** = |SOC reportado − SOC referencia| → discrepancia de estado de carga.
- **Residuo de potencia aparente** = √(P² + Q²) − S_medido por medidor → consistencia interna del medidor.
- **Desequilibrio de fase** = max |Vᵢ − Vⱼ| por medidor → calidad de tensión.

In [ ]:
# --- Invariantes ciberfísicas: quitan el nivel absoluto y dejan la física ---
MEDIDORES = [
    "BSS-Inverter-AC-Meter", "Load-A-AC-Meter", "Load-B-AC-Meter",
    "PV-Inverter-AC-Meter", "Substation-AC-Meter",
]


def invariantes_fisicas(df):
    """Construye las tres familias de invariantes a partir de un bloque."""
    inv = pd.DataFrame(index=df.index)

    # ΔSOC = |SOC reportado - SOC de referencia|
    inv["dsoc"] = (df["Battery-SOC-Reported.soc_value"]
                   - df["Battery-SOC-Reference.soc_value"]).abs()

    for medidor in MEDIDORES:
        # Residuo de potencia aparente: sqrt(P^2 + Q^2) - S_medido
        # (si ronda 0, el medidor es internamente consistente).
        p = df[f"{medidor}.p_sum3"]
        q = df[f"{medidor}.q_sum3"]
        s = df[f"{medidor}.s_sum3"]
        inv[f"{medidor}__serror"] = np.sqrt(p**2 + q**2) - s

        # Desequilibrio de fase: max |V_i - V_j| entre las tres fases.
        fases = [df[f"{medidor}.uln1"], df[f"{medidor}.uln2"], df[f"{medidor}.uln3"]]
        pares = [(fases[i] - fases[j]).abs() for i in range(3) for j in range(i + 1, 3)]
        inv[f"{medidor}__imbalance"] = pd.concat(pares, axis=1).max(axis=1)

    return inv


# Se apilan los ocho escenarios y se alinean etiqueta y escenario.
X_inv = pd.concat([invariantes_fisicas(X[k]) for k in ESCENARIOS], ignore_index=True)
y = np.concatenate([Y[k].to_numpy() for k in ESCENARIOS])
escenario = np.concatenate([np.full(len(Y[k]), k) for k in ESCENARIOS])

print("matriz de invariantes:", X_inv.shape)
display(X_inv.head())

## 3. Pipeline y elección de K

`imputar → RobustScaler → PCA(0.95) → KMeans`, con K elegido por silueta (y controlado con Davies-Bouldin).

In [ ]:
# --- Pipeline: imputar -> escalado robusto -> PCA.
# RobustScaler (mediana + IQR) porque los invariantes tienen colas pesadas:
# StandardScaler se infla con los outliers y aplasta la estructura real. ---
def preprocesar(X):
    return make_pipeline(
        SimpleImputer(strategy="median"),
        RobustScaler(),
        PCA(n_components=0.95, random_state=SEED),
    ).fit_transform(X)


Z = preprocesar(X_inv)

# --- Elección de K: silueta (arriba mejor) y Davies-Bouldin (abajo mejor), K = 2..10 ---
filas = []
for k in range(2, 11):
    etiquetas = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(Z)
    filas.append({
        "k": k,
        "silueta": silhouette_score(Z, etiquetas, sample_size=3000, random_state=SEED),
        "davies_bouldin": davies_bouldin_score(Z, etiquetas),
    })
tabla_k = pd.DataFrame(filas).set_index("k").round(3)
display(tabla_k)

print("K elegido por silueta:", int(tabla_k["silueta"].idxmax()))

In [ ]:
# --- Visual de la elección de K ---
fig, ejes = plt.subplots(1, 2, figsize=(9, 3))
ejes[0].plot(tabla_k.index, tabla_k["silueta"], marker="o")
ejes[0].set_title("silueta (arriba = mejor)")
ejes[0].set_xlabel("k")
ejes[1].plot(tabla_k.index, tabla_k["davies_bouldin"], marker="o", color="darkorange")
ejes[1].set_title("Davies-Bouldin (abajo = mejor)")
ejes[1].set_xlabel("k")
plt.tight_layout()
plt.show()

## 4. Perfil de los clusters (z-score)

Un |z| alto indica que esa variable es la que **define** el cluster, porque se aparta de la media global:
`z = (media del cluster − media global) / desvío global`.

In [ ]:
# --- Perfil de los clusters con z-score del centroide ---
K = int(tabla_k["silueta"].idxmax())
etiquetas = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit_predict(Z)

valores = X_inv.to_numpy(dtype=float)
media_global = valores.mean(axis=0)
desvio_global = valores.std(axis=0)
desvio_global[desvio_global == 0] = 1


def describir(variable):
    """Traduce el nombre interno de un invariante a algo legible."""
    if variable == "dsoc":
        return "desviación de SOC"
    medidor, _, tipo = variable.partition("__")
    nombre = {"serror": "residuo de potencia aparente",
              "imbalance": "desequilibrio de fase"}[tipo]
    return f"{medidor} · {nombre}"


# Las 3 variables que más definen cada cluster (mayor |z|).
filas = []
for c in range(K):
    en_cluster = etiquetas == c
    z = (valores[en_cluster].mean(axis=0) - media_global) / desvio_global
    for j in np.argsort(-np.abs(z))[:3]:
        filas.append({
            "cluster": c,
            "n": int(en_cluster.sum()),
            "variable": describir(X_inv.columns[j]),
            "z": round(float(z[j]), 1),
        })
display(pd.DataFrame(filas))

# --- Resumen: métricas del clustering elegido + mezcla de escenario y ataque ---
print("silueta:", round(silhouette_score(Z, etiquetas, sample_size=3000, random_state=SEED), 3))
print("Davies-Bouldin:", round(davies_bouldin_score(Z, etiquetas), 3))
print("ARI vs ataque:", round(adjusted_rand_score(y, etiquetas), 3))
print("ARI vs escenario:", round(adjusted_rand_score(escenario, etiquetas), 3))
display(pd.crosstab(pd.Series(etiquetas, name="cluster"),
                    pd.Series(escenario, name="escenario")))